In [3]:
import pandas as pd
import numpy as np
import time
from typing import Optional

In [14]:
class SimpleBatteryStorage:
    def __init__(self, states:np.ndarray, actions:np.ndarray, prices:np.ndarray, start_state:int, end_state:int):
        self.states = states.reshape(1, -1)
        self.actions = actions.reshape(1, -1)
        self.prices = prices.reshape(-1, 1)
        self.start_state = start_state
        self.end_state = end_state
        self.number_states = self.states.shape[-1]
        
    def get_state_transitions(self):
        state_base = (np.ones(shape=(self.states.shape[-1], self.actions.shape[-1])) * self.states.reshape(-1,1)).reshape(-1, 1)
        actions = np.tile(self.actions.T,(state_base.shape[0]//self.actions.T.shape[0],1))
        next_state = state_base + actions
        timestep_state_transition = np.concat([state_base, actions, next_state], axis=1)

        actions_clmn = timestep_state_transition[:,1][..., np.newaxis]
        prices = self.prices.T
        costs = (prices * actions_clmn).T.reshape(-1,1) 
        
        tiling = [1]*timestep_state_transition.shape[-1]
        tiling[0] = costs.shape[0]//timestep_state_transition.shape[0]
        state_transition = np.concat([np.tile(timestep_state_transition, reps=tiling).reshape(-1, 3), costs], axis=1)
        discharge_violation_ids = np.argwhere(state_transition[:, 2] < np.min(self.states)).squeeze()
        charge_violation_ids = np.argwhere(state_transition[:, 2] > np.max(self.states)).squeeze()
        
        state_transition[discharge_violation_ids, 2] = np.min(self.states)
        state_transition[charge_violation_ids, 2] = np.max(self.states)
        
        state_transition[np.concat([charge_violation_ids, discharge_violation_ids]), -1] = np.inf
        
        time_ids = np.tile(np.array(range(len(self.prices))), reps=[state_transition.shape[0]//len(self.prices),1])
        
        state_transition_matrix = np.concatenate([time_ids.T.reshape(-1, 1), state_transition], axis=1)
        return state_transition_matrix
    




class SimpleBatteryStorageChargesRestricted:
    def __init__(self, states:np.ndarray, actions:np.ndarray, prices:np.ndarray, start_state:int, end_state:int, max_charges:Optional[int]):
        self.states = states.reshape(-1, 1)
        self.actions = actions.reshape(1, -1)
        self.prices = prices
        self.start_state = start_state
        self.end_state = end_state
        self.max_charges = max_charges
        self.number_states = self.states.shape[0] * (self.max_charges+1) if max_charges is not None else self.states.shape[0]
        self.state_mapping = None
        
    def get_state_transitions(self):
        if self.max_charges is not None:
            skeleton_array = np.ones(shape=(self.states.shape[-1]*(self.max_charges+1)*self.actions.shape[-1]))
        else:
            skeleton_array = np.ones(shape=(self.states.shape[-1]*self.actions.shape[-1]))
            
        state_base = (skeleton_array*self.states).reshape(-1,1)

        charges_base = np.tile(np.arange(0, self.max_charges+1).reshape(-1,1), (self.states.shape[0], self.actions.shape[-1])).reshape(-1,1)
        _actions = np.resize(self.actions, (charges_base.shape[0], 1))

        next_state = np.concat([state_base+_actions, charges_base+np.abs(_actions)], axis=1)
        timestep_state_transition = np.concat([state_base, charges_base, _actions, next_state], axis=1)

        actions_clmn_id = 2

        actions_clmn = timestep_state_transition[:,actions_clmn_id][..., np.newaxis]
        prices = self.prices.T
        costs = (prices * actions_clmn).T.reshape(-1,1)

        state_transition = np.resize(timestep_state_transition, (costs.shape[0], timestep_state_transition.shape[1]))

        discharge_violation_ids = np.argwhere(state_transition[:, actions_clmn_id+1] < np.min(self.states)).squeeze()
        charge_violation_ids = np.argwhere(state_transition[:, actions_clmn_id+1] > np.max(self.states)).squeeze()

        charges_violation_ids = np.argwhere(state_transition[:, actions_clmn_id+2] > self.max_charges).squeeze()

        state_transition[discharge_violation_ids, actions_clmn_id+1] = np.min(self.states)
        state_transition[charge_violation_ids, actions_clmn_id+1] = np.max(self.states)
        state_transition[charges_violation_ids, actions_clmn_id+2] = self.max_charges

        _state_transition = np.concat([state_transition, costs], axis=1)
        state_transition = np.delete(_state_transition, np.unique(np.concat([discharge_violation_ids, charge_violation_ids, charges_violation_ids])), axis=0)
        #state_transition[np.unique(np.concat([discharge_violation_ids, charge_violation_ids, charges_violation_ids])), -1] = np.inf
        #time_ids = np.tile(np.array(range(len(prices))), reps=[state_transition.shape[0]//len(prices),1])
        
        #state_transition_matrix = np.concatenate([time_ids.T.reshape(-1, 1), state_transition], axis=1)
        #return state_transition_matrix
        return state_transition
    

    
    def map_states(self, state_transition_matrix):
        # Extract current and next state pairs
        current_states = state_transition_matrix[:, [0, 1]]
        next_states = state_transition_matrix[:, [3, 4]]
        
        # Combine both for unified unique indexing
        all_states = np.vstack((current_states, next_states))
        
        # Get unique states and their indices
        start = time.time()
        
        #unique_states, inverse_indices = np.unique(all_states, axis=0, return_inverse=True)
        # np.unique is slower than using pandas here
        df_states = pd.DataFrame(all_states).astype(int)
        unique_states_df = df_states.drop_duplicates().reset_index(drop=True).reset_index()
        inverse_indices = df_states.merge(unique_states_df, how='left', on=[0,1], sort=False)['index'].to_numpy()
        end = time.time()
        print(f'UNIQUE: {end-start}')
        
        start = time.time()
        #self.state_mapping = {tuple(state): idx for idx, state in enumerate(unique_states)} 
        #self.state_mapping = {tuple(state): idx for idx, state in enumerate(unique_states_df.loc[:, [0, 1]].to_numpy())}
        #self.state_mapping = {(state.0, state.1): state.index for _, state in unique_states_df.iterrows()}
        self.state_mapping = pd.Series(unique_states_df['index'].values, index=list(zip(unique_states_df[0], unique_states_df[1]))).to_dict()
        end = time.time()
        print(f'MAPPING DICT: {end-start}')
        
        #self.state_mapping = run_mapping(unique_states=unique_states)
        # Split back the indices
        current_state_ids = inverse_indices[:len(current_states)]
        next_state_ids = inverse_indices[len(current_states):]
        
        # Construct the final mapped matrix
        return np.column_stack((
            current_state_ids,
            state_transition_matrix[:, 2],
            next_state_ids, 
            state_transition_matrix[:, 5:]
        ))
        


In [15]:
prices = np.array([1,5])
prices = np.random.uniform(low=1, high=10, size=8760)
states = np.array([0, 1])
actions = np.array([-1, 0, 1])
bs = SimpleBatteryStorageChargesRestricted(states=states, actions=actions, prices=prices, start_state=0, end_state=0, max_charges=200)
bs_matrix = bs.map_states(bs.get_state_transitions())
bs_matrix = bs_matrix.reshape((len(prices), bs_matrix.shape[0]//len(prices), bs_matrix.shape[-1]))
bs_matrix

UNIQUE: 2.614309310913086
MAPPING DICT: 0.0028972625732421875


array([[[  0.        ,   0.        ,   0.        ,   0.        ],
        [  0.        ,   1.        , 202.        ,   4.61084333],
        [  1.        ,   0.        ,   1.        ,   0.        ],
        ...,
        [400.        ,  -1.        , 200.        ,  -4.61084333],
        [400.        ,   0.        , 400.        ,   0.        ],
        [401.        ,   0.        , 401.        ,   0.        ]],

       [[  0.        ,   0.        ,   0.        ,   0.        ],
        [  0.        ,   1.        , 202.        ,   5.5012856 ],
        [  1.        ,   0.        ,   1.        ,   0.        ],
        ...,
        [400.        ,  -1.        , 200.        ,  -5.5012856 ],
        [400.        ,   0.        , 400.        ,   0.        ],
        [401.        ,   0.        , 401.        ,   0.        ]],

       [[  0.        ,   0.        ,   0.        ,   0.        ],
        [  0.        ,   1.        , 202.        ,   8.89386153],
        [  1.        ,   0.        ,   1.     

In [16]:

bs_matrix.shape


(8760, 802, 4)

In [17]:
# backward
value_list = []
for i,t in enumerate(reversed(range(bs_matrix.shape[0]))):
    if i == 0:
        previous_state_values = np.zeros(shape=(1, bs.number_states))
    else:
        previous_state_values = value_list[i-1]
    
    # t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    # temp_array = bs_matrix[t_id,...]
    
    temp_array = bs_matrix[t, ...]
    
    temp_array = temp_array[np.argsort(temp_array[:, 0]), ...]
    
    temp_state_values = previous_state_values[:, np.int32(temp_array[:, -2])].squeeze()

    temp_state_values = temp_state_values + temp_array[:, -1]
    
    unique_states, idx_start = np.unique(temp_array[:, 0].astype(int), return_index=True)
    value_list.append(np.minimum.reduceat(temp_state_values, idx_start).reshape((1, bs.number_states)))
    #value_list.append(np.min(temp_state_values.reshape(bs.number_states, -1), axis=1).reshape((1, bs.number_states)))

In [18]:
value_list

[array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.

In [19]:
# forward 
forward_value_list = list(reversed(value_list))
chosen_states = []
chosen_actions = []
for t in range(bs_matrix.shape[0]):
    # t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t,...]
    
    if t == 0:
        if (bs.start_state is not None):
            temp_state = bs.start_state
        
        else:
            temp_state = np.argmin(forward_value_list[t].squeeze())
        chosen_states.append(temp_state)
    else:
        temp_state = chosen_states[t] # not t-1 since we already have the initial state in the chosen states list

    temp_array = temp_array[np.argwhere(temp_array[:, 0]==temp_state).squeeze(), ...].reshape((-1, temp_array.shape[-1]))
    
    if t == len(prices)-1:
        next_state_value = np.zeros(shape=(1, bs.number_states))
    else:
        next_state_value = forward_value_list[t+1]
    
    costs = temp_array[:, -1]
    diff_arr = (next_state_value[:, np.int32(temp_array[:, -2])] + costs).squeeze()
    
    action_selection = np.argmin(diff_arr).squeeze()
    
    chosen_actions.append(int(temp_array[action_selection, 1]))
    chosen_states.append(int(temp_array[action_selection, 2]))

In [20]:
chosen_states[-1]

200

In [21]:
chosen_actions

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 -1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,

In [22]:
bs.state_mapping


{(0, 0): 0,
 (0, 1): 1,
 (0, 2): 2,
 (0, 3): 3,
 (0, 4): 4,
 (0, 5): 5,
 (0, 6): 6,
 (0, 7): 7,
 (0, 8): 8,
 (0, 9): 9,
 (0, 10): 10,
 (0, 11): 11,
 (0, 12): 12,
 (0, 13): 13,
 (0, 14): 14,
 (0, 15): 15,
 (0, 16): 16,
 (0, 17): 17,
 (0, 18): 18,
 (0, 19): 19,
 (0, 20): 20,
 (0, 21): 21,
 (0, 22): 22,
 (0, 23): 23,
 (0, 24): 24,
 (0, 25): 25,
 (0, 26): 26,
 (0, 27): 27,
 (0, 28): 28,
 (0, 29): 29,
 (0, 30): 30,
 (0, 31): 31,
 (0, 32): 32,
 (0, 33): 33,
 (0, 34): 34,
 (0, 35): 35,
 (0, 36): 36,
 (0, 37): 37,
 (0, 38): 38,
 (0, 39): 39,
 (0, 40): 40,
 (0, 41): 41,
 (0, 42): 42,
 (0, 43): 43,
 (0, 44): 44,
 (0, 45): 45,
 (0, 46): 46,
 (0, 47): 47,
 (0, 48): 48,
 (0, 49): 49,
 (0, 50): 50,
 (0, 51): 51,
 (0, 52): 52,
 (0, 53): 53,
 (0, 54): 54,
 (0, 55): 55,
 (0, 56): 56,
 (0, 57): 57,
 (0, 58): 58,
 (0, 59): 59,
 (0, 60): 60,
 (0, 61): 61,
 (0, 62): 62,
 (0, 63): 63,
 (0, 64): 64,
 (0, 65): 65,
 (0, 66): 66,
 (0, 67): 67,
 (0, 68): 68,
 (0, 69): 69,
 (0, 70): 70,
 (0, 71): 71,
 (0, 72): 72

In [25]:
pd.DataFrame.from_dict(bs.state_mapping.keys(), index=bs.state_mapping.values())

TypeError: DataFrame.from_dict() got an unexpected keyword argument 'index'

In [31]:
reverse_df = pd.concat([pd.Series(bs.state_mapping.keys(), name='States'), pd.Series(bs.state_mapping.values(), name='Mapped States')], axis=1)
reverse_df


,States,Mapped States
0,"(0, 0)",0
1,"(0, 1)",1
2,"(0, 2)",2
3,"(0, 3)",3
4,"(0, 4)",4
...,...,...
397,"(1, 196)",397
398,"(1, 197)",398
399,"(1, 198)",399
400,"(1, 199)",400


In [37]:
reverse_df.iloc[chosen_states, :]['States'].to_list()

[(0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 

In [23]:
np.sum(np.array(chosen_actions) * prices)

np.float64(-886.394749394439)

In [74]:
prices = np.array([1,5,3,5])
#prices = np.random.uniform(low=1, high=10, size=8760)
states = np.array([0, 1])
actions = np.array([-1, 0, 1])
bs = SimpleBatteryStorage(states=states, actions=actions, prices=prices, start_state=0, end_state=0)
bs_matrix = bs.get_state_transitions()
bs_matrix

array([[ 0.,  0., -1.,  0., inf],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  1.,  1.,  1.],
       [ 0.,  1., -1.,  0., -1.],
       [ 0.,  1.,  0.,  1.,  0.],
       [ 0.,  1.,  1.,  1., inf],
       [ 1.,  0., -1.,  0., inf],
       [ 1.,  0.,  0.,  0.,  0.],
       [ 1.,  0.,  1.,  1.,  5.],
       [ 1.,  1., -1.,  0., -5.],
       [ 1.,  1.,  0.,  1.,  0.],
       [ 1.,  1.,  1.,  1., inf],
       [ 2.,  0., -1.,  0., inf],
       [ 2.,  0.,  0.,  0.,  0.],
       [ 2.,  0.,  1.,  1.,  3.],
       [ 2.,  1., -1.,  0., -3.],
       [ 2.,  1.,  0.,  1.,  0.],
       [ 2.,  1.,  1.,  1., inf],
       [ 3.,  0., -1.,  0., inf],
       [ 3.,  0.,  0.,  0.,  0.],
       [ 3.,  0.,  1.,  1.,  5.],
       [ 3.,  1., -1.,  0., -5.],
       [ 3.,  1.,  0.,  1.,  0.],
       [ 3.,  1.,  1.,  1., inf]])

In [80]:
# backward
value_list = []
number_state_action_pairs = bs_matrix.shape[0]//len(prices)
for i,t in enumerate(reversed(range(len(prices)))):
    if i == 0:
        previous_state_values = np.zeros(shape=(1, bs.states.shape[-1]))
    else:
        previous_state_values = value_list[i-1]
    
    t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t_id,...]
    #print(temp_array)
    temp_state_values = previous_state_values[:, np.int32(temp_array[:, -2])].squeeze()

    temp_state_values = temp_state_values + temp_array[:, -1]
    
    value_list.append(np.min(temp_state_values.reshape(bs.states.shape[-1], -1), axis=1).reshape((1, bs.states.shape[-1])))
    
    

In [76]:
value_list

[array([[ 0., -5.]]),
 array([[-2., -5.]]),
 array([[-2., -7.]]),
 array([[-6., -7.]])]

In [77]:
# forward 
forward_value_list = list(reversed(value_list))
chosen_states = []
chosen_actions = []
for t in range(len(prices)):
    t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t_id,...]
    
    if t == 0:
        if (bs.start_state is not None):
            temp_state = bs.start_state
        
        else:
            temp_state = np.argmin(forward_value_list[t].squeeze())
        chosen_states.append(temp_state)
    else:
        temp_state = chosen_states[t] # not t-1 since we already have the initial state in the chosen states list

    temp_array = temp_array[np.argwhere(temp_array[:, 1]==temp_state).squeeze(), ...]
    
    if t == len(prices)-1:
        next_state_value = np.zeros(shape=(1, bs.states.shape[-1]))
    else:
        next_state_value = forward_value_list[t+1]
    
    costs = temp_array[:, -1]
    diff_arr = (next_state_value[:, np.int32(temp_array[:, -2])] + costs).squeeze()
    
    action_selection = np.argmin(diff_arr).squeeze()
    
    chosen_actions.append(int(temp_array[action_selection, 2]))
    chosen_states.append(int(temp_array[action_selection, 3]))
    
    


In [78]:
chosen_states

[0, 1, 0, 1, 0]

In [79]:
chosen_actions

[1, -1, 1, -1]

In [9]:
np.sum(np.array(chosen_actions) * prices)

np.float64(-17729.731026335616)